In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
import warnings
warnings.filterwarnings("ignore")

In [ ]:
train_df=pd.read_csv('data/train.csv')
test_df=pd.read_csv('data/test.csv')

In [ ]:
train_df.head(3)

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288585,-0.020294,-0.132905,-0.995279,-0.983111,-0.913526,-0.995112,-0.983185,-0.923527,-0.934724,...,-0.710304,-0.112754,0.030400,-0.464761,-0.018446,-0.841247,0.179941,-0.058627,1,STANDING
1,0.278419,-0.016411,-0.123520,-0.998245,-0.975300,-0.960322,-0.998807,-0.974914,-0.957686,-0.943068,...,-0.861499,0.053477,-0.007435,-0.732626,0.703511,-0.844788,0.180289,-0.054317,1,STANDING
2,0.279653,-0.019467,-0.113462,-0.995380,-0.967187,-0.978944,-0.996520,-0.963668,-0.977469,-0.938692,...,-0.760104,-0.118559,0.177899,0.100699,0.808529,-0.848933,0.180637,-0.049118,1,STANDING


In [ ]:
test_df.head(3)

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.257178,-0.023285,-0.014654,-0.938404,-0.920091,-0.667683,-0.952501,-0.925249,-0.674302,-0.894088,...,-0.705974,0.006462,0.162920,-0.825886,0.271151,-0.720009,0.276801,-0.057978,2,STANDING
1,0.286027,-0.013163,-0.119083,-0.975415,-0.967458,-0.944958,-0.986799,-0.968401,-0.945823,-0.894088,...,-0.594944,-0.083495,0.017500,-0.434375,0.920593,-0.698091,0.281343,-0.083898,2,STANDING
2,0.275485,-0.026050,-0.118152,-0.993819,-0.969926,-0.962748,-0.994403,-0.970735,-0.963483,-0.939260,...,-0.640736,-0.034956,0.202302,0.064103,0.145068,-0.702771,0.280083,-0.079346,2,STANDING


In [ ]:
train_df.isnull().sum()

,0
tBodyAcc-mean()-X,0
tBodyAcc-mean()-Y,0
tBodyAcc-mean()-Z,0
tBodyAcc-std()-X,0
tBodyAcc-std()-Y,0
...,...
"angle(X,gravityMean)",0
"angle(Y,gravityMean)",0
"angle(Z,gravityMean)",0
subject,0


In [ ]:
test_df.isnull().sum()

,0
tBodyAcc-mean()-X,0
tBodyAcc-mean()-Y,0
tBodyAcc-mean()-Z,0
tBodyAcc-std()-X,0
tBodyAcc-std()-Y,0
...,...
"angle(X,gravityMean)",0
"angle(Y,gravityMean)",0
"angle(Z,gravityMean)",0
subject,0


In [ ]:
X_train = train_df.drop(['Activity', 'subject'], axis=1)
y_train = train_df['Activity']
X_test = test_df.drop(['Activity', 'subject'] , axis=1)
y_test = test_df['Activity']

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [ ]:
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('lda', LDA()),
    ('model', RandomForestClassifier())
])

pipe_svc = Pipeline([
    ('scaler', StandardScaler()),
    ('lda', LDA()),
    ('model', SVC())
])

pipe_knc = Pipeline([
    ('scaler', StandardScaler()),
    ('lda', LDA()),
    ('model', KNeighborsClassifier())
])


In [ ]:
model_hyperparameters = {


    pipe_rf: {
        'model__n_estimators': [50,100],
        'model__max_depth': [None, 10, 20]

    },


    pipe_svc: {
        'model__C': [0.1, 1, 10, 100],
        'model__kernel': ['linear', 'rbf']

    },


    pipe_knc: {
        'model__n_neighbors': [3, 5, 7, 9, 11],

        },


}

In [ ]:
def ModelSelection(model_hyperparameters, X_train, y_train):
    results = []
    best_model = None
    best_score = -np.inf  # Initialize best_score to negative infinity

    for model, params in model_hyperparameters.items():

        grid = GridSearchCV(model, params, cv=3, scoring='accuracy')
        grid.fit(X_train, y_train)
        score = grid.best_score_

        results.append({
            "model": model,
            "best_score": score,
            "best_params": grid.best_params_
        })

        if score > best_score:
            best_score = score
            best_model = grid.best_estimator_

    results_df = pd.DataFrame(results)
    return results_df, best_model

In [ ]:
results_df, best_model = ModelSelection(model_hyperparameters, X_train, y_train)
print(results_df)

print("Best Model:", best_model)

                                               model  best_score  \
0  (StandardScaler(), LinearDiscriminantAnalysis(...    0.947501   
1  (StandardScaler(), LinearDiscriminantAnalysis(...    0.953620   
2  (StandardScaler(), LinearDiscriminantAnalysis(...    0.951716   

                                         best_params  
0  {'model__max_depth': 20, 'model__n_estimators'...  
1            {'model__C': 1, 'model__kernel': 'rbf'}  
2                         {'model__n_neighbors': 11}  
Best Model: Pipeline(steps=[('scaler', StandardScaler()),
                ('lda', LinearDiscriminantAnalysis()), ('model', SVC(C=1))])


In [ ]:
print("Best Model:", best_model)

Best Model: Pipeline(steps=[('scaler', StandardScaler()),
                ('lda', LinearDiscriminantAnalysis()), ('model', SVC(C=1))])


In [ ]:
y_pred = best_model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

print("Test Accuracy:", accuracy_score(y_test, y_pred))

Test Accuracy: 0.9650492025788938


In [ ]:
y_pred

array([2, 2, 2, ..., 5, 5, 5])

In [ ]:
y_test

array([2, 2, 2, ..., 5, 5, 5])